# Analisi del Campo "reason" - Decision Making Analysis

Questo notebook analizza il campo "reason" nei risultati delle predizioni per comprendere:
1. **Fattori decisionali** utilizzati dai modelli LLM
2. **Qualità del reasoning** e correlazione con performance
3. **Differenze tra versioni** (base, geom, geom_time)
4. **Pattern linguistici** e strutture argomentative

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict
import re
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Parallelizzazione e monitoring
from multiprocessing import Pool, cpu_count
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, as_completed
import time
from tqdm import tqdm
tqdm.pandas(desc="Processing")

# Setup NLTK con gestione certificati SSL e fallback robusti
import ssl
print("🔧 Configurando ambiente per analisi reasoning...")

# Fix per certificati SSL su macOS/Linux
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

# Import NLTK con fallback completo
NLTK_AVAILABLE = False
try:
    import nltk
    print("✓ NLTK importato con successo")
    
    # Download required NLTK data con gestione errori
    def setup_nltk():
        """Setup NLTK data con fallback"""
        nltk_downloads = [
            ('punkt', 'tokenizers/punkt'),
            ('punkt_tab', 'tokenizers/punkt_tab'), 
            ('stopwords', 'corpora/stopwords')
        ]

        for package, path in nltk_downloads:
            try:
                nltk.data.find(path)
                print(f"✓ {package} già disponibile")
            except LookupError:
                try:
                    print(f"⬇️  Downloading {package}...")
                    nltk.download(package, quiet=True)
                    print(f"✅ {package} scaricato")
                except Exception as e:
                    print(f"⚠️  Errore scaricamento {package}: {e}")

    setup_nltk()

    # Test import tokenizers
    try:
        from nltk.corpus import stopwords
        from nltk.tokenize import word_tokenize, sent_tokenize
        NLTK_AVAILABLE = True
        print("✅ NLTK tokenizers disponibili")
    except ImportError as e:
        print(f"⚠️  NLTK tokenizers non disponibili: {e}")
        NLTK_AVAILABLE = False
        
except ImportError as e:
    print(f"⚠️  NLTK non disponibile: {e}")
    print("📝 Utilizzerò tokenizzazione semplificata")
    NLTK_AVAILABLE = False

# Import textstat con fallback
TEXTSTAT_AVAILABLE = False
try:
    from textstat import flesch_reading_ease, flesch_kincaid_grade
    TEXTSTAT_AVAILABLE = True
    print("✅ TextStat disponibile")
except ImportError:
    print("⚠️  TextStat non disponibile - metriche di leggibilità disabilitate")
    TEXTSTAT_AVAILABLE = False

# Import wordcloud con fallback
WORDCLOUD_AVAILABLE = False
try:
    from wordcloud import WordCloud
    WORDCLOUD_AVAILABLE = True
    print("✅ WordCloud disponibile")
except ImportError:
    print("⚠️  WordCloud non disponibile - word clouds disabilitate")
    WORDCLOUD_AVAILABLE = False

# Import psutil con fallback
PSUTIL_AVAILABLE = False
try:
    import psutil
    PSUTIL_AVAILABLE = True
    print("✅ PSUtil disponibile")
except ImportError:
    print("⚠️  PSUtil non disponibile - monitoraggio sistema disabilitato")
    PSUTIL_AVAILABLE = False

# Configurazione parallelizzazione
N_CORES = min(cpu_count(), 8)  # Limita a 8 core per non sovraccaricare
CHUNK_SIZE = 10000  # Dimensione ottimale dei chunks

print(f"✅ Configurazione parallelizzazione: {N_CORES} cores, chunks di {CHUNK_SIZE}")

# Configurazione plot
plt.rcParams['figure.figsize'] = (12, 8)
sns.set_palette("husl")

print("✅ Setup completato!")
print(f"📊 Funzionalità disponibili:")
print(f"   • NLTK: {'✅' if NLTK_AVAILABLE else '❌'}")
print(f"   • TextStat: {'✅' if TEXTSTAT_AVAILABLE else '❌'}")
print(f"   • WordCloud: {'✅' if WORDCLOUD_AVAILABLE else '❌'}")
print(f"   • PSUtil: {'✅' if PSUTIL_AVAILABLE else '❌'}")
print("🚀 Ready to analyze reasoning data!")

## 1. Caricamento e Preparazione Dati

In [ ]:
def load_all_csv_files(results_dir="../results"):
    """
    Carica tutti i file CSV dai risultati e li organizza per modello e strategia
    """
    all_data = []
    
    results_path = Path(results_dir)
    
    for csv_file in results_path.rglob("*.csv"):
        if "checkpoint" not in csv_file.name:
            try:
                # Estrai informazioni dal path
                parts = csv_file.parts
                
                # Determina anchor, model, strategy
                anchor = "last"  # default
                if "penultimate" in parts:
                    anchor = "penultimate"
                elif "middle" in parts:
                    anchor = "middle"
                
                # Trova modello e strategia
                model = None
                strategy = None
                
                for i, part in enumerate(parts):
                    if any(m in part for m in ['qwen', 'mistral', 'deepseek', 'llama']):
                        model = part
                        if i + 1 < len(parts):
                            strategy = parts[i + 1]
                        break
                
                if not model or not strategy:
                    continue
                    
                # Carica CSV
                df = pd.read_csv(csv_file)
                
                # Aggiungi metadati
                df['source_file'] = csv_file.name
                df['model'] = model
                df['strategy'] = strategy
                df['anchor'] = anchor
                
                all_data.append(df)
                print(f"✓ Loaded {csv_file.name} - {model}/{strategy}/{anchor} - {len(df)} rows")
                
            except Exception as e:
                print(f"✗ Error loading {csv_file.name}: {e}")
    
    if all_data:
        combined_df = pd.concat(all_data, ignore_index=True)
        print(f"\n✓ Total loaded: {len(combined_df)} predictions from {len(all_data)} files")
        return combined_df
    else:
        print("✗ No CSV files found")
        return pd.DataFrame()

# Carica tutti i dati
df = load_all_csv_files()

# Informazioni di base
print(f"\nDimensioni dataset: {df.shape}")
print(f"Modelli: {df['model'].unique()}")
print(f"Strategie: {df['strategy'].unique()}")
print(f"Anchor points: {df['anchor'].unique()}")

In [ ]:
# Filtra solo le righe con status 'success' per l'analisi del reasoning
df_success = df[df['status'] == 'success'].copy()
df_success = df_success[df_success['reason'].notna()].copy()
df_success = df_success[df_success['reason'].str.strip() != ''].copy()

print(f"Predizioni con reasoning valido: {len(df_success)} / {len(df)} ({100*len(df_success)/len(df):.1f}%)")

# Overview delle performance generali
performance_by_category = df_success.groupby(['model', 'strategy']).agg({
    'hit': ['count', 'sum', 'mean'],
    'processing_time': 'mean'
}).round(3)

performance_by_category.columns = ['total_predictions', 'hits', 'hit_rate', 'avg_processing_time']
print("\nPerformance per modello/strategia:")
print(performance_by_category)

## 2. Analisi Quantitativa del Reasoning

In [ ]:
def extract_reasoning_features(reason_text):
    """
    Estrae features quantitative dal testo del reasoning con fallback per NLTK
    """
    if pd.isna(reason_text) or reason_text.strip() == '':
        return {
            'length_chars': 0,
            'length_words': 0,
            'num_sentences': 0,
            'avg_word_length': 0,
            'flesch_score': 0,
            'has_geographic_terms': False,
            'has_temporal_terms': False,
            'has_popularity_terms': False,
            'has_distance_terms': False,
            'reasoning_complete': False
        }
    
    text = str(reason_text).strip()
    
    # Features di base
    length_chars = len(text)
    
    # Fallback per tokenizzazione se NLTK non è disponibile
    if NLTK_AVAILABLE:
        try:
            words = word_tokenize(text.lower())
            sentences = sent_tokenize(text)
        except:
            # Fallback semplice se NLTK fallisce
            words = text.lower().split()
            sentences = text.split('.')
    else:
        # Tokenizzazione semplice senza NLTK
        words = text.lower().split()
        sentences = text.split('.')
    
    length_words = len(words)
    num_sentences = len([s for s in sentences if s.strip()])
    
    # Lunghezza media parole
    alpha_words = [w for w in words if w.isalpha()]
    avg_word_length = np.mean([len(w) for w in alpha_words]) if alpha_words else 0
    
    # Readability score
    if TEXTSTAT_AVAILABLE:
        try:
            flesch_score = flesch_reading_ease(text)
        except:
            flesch_score = 0
    else:
        flesch_score = 0
    
    # Terms semantici
    geographic_terms = ['close', 'near', 'distance', 'walking', 'vicinity', 'nearby', 'location', 'area']
    temporal_terms = ['time', 'hour', 'morning', 'afternoon', 'evening', 'before', 'after', 'dark', 'late']
    popularity_terms = ['popular', 'famous', 'well-known', 'major', 'attraction', 'tourist', 'visited']
    distance_terms = ['close', 'near', 'far', 'distance', 'km', 'meter', 'walk', 'reasonable']
    
    text_lower = text.lower()
    has_geographic = any(term in text_lower for term in geographic_terms)
    has_temporal = any(term in text_lower for term in temporal_terms)
    has_popularity = any(term in text_lower for term in popularity_terms)
    has_distance = any(term in text_lower for term in distance_terms)
    
    # Reasoning completeness (euristica)
    reasoning_complete = (
        length_chars > 50 and 
        num_sentences >= 1 and 
        not text.endswith('..') and
        not 'truncated' in text_lower
    )
    
    return {
        'length_chars': length_chars,
        'length_words': length_words,
        'num_sentences': num_sentences,
        'avg_word_length': avg_word_length,
        'flesch_score': flesch_score,
        'has_geographic_terms': has_geographic,
        'has_temporal_terms': has_temporal,
        'has_popularity_terms': has_popularity,
        'has_distance_terms': has_distance,
        'reasoning_complete': reasoning_complete
    }

def threaded_feature_extraction(series, n_threads=N_CORES, chunk_size=CHUNK_SIZE):
    """
    Estrae features con ThreadPoolExecutor (compatibile Jupyter)
    """
    print(f"🚀 Avvio estrazione features con threading:")
    print(f"   • Dataset: {len(series):,} predizioni")
    print(f"   • Threads: {n_threads}")
    print(f"   • Chunk size: {chunk_size:,}")
    
    start_time = time.time()
    
    # Dividi in chunks
    chunks = [series[i:i+chunk_size] for i in range(0, len(series), chunk_size)]
    print(f"   • Chunks creati: {len(chunks)}")
    
    # Funzione per processare chunk con threading
    def process_chunk_threaded(chunk):
        return [extract_reasoning_features(text) for text in chunk]
    
    # Processa con ThreadPoolExecutor
    results = []
    with ThreadPoolExecutor(max_workers=n_threads) as executor:
        # Invia tutti i chunks
        future_to_chunk = {executor.submit(process_chunk_threaded, chunk): i 
                          for i, chunk in enumerate(chunks)}
        
        # Monitora progressi
        with tqdm(total=len(chunks), desc="Processing chunks") as pbar:
            for future in as_completed(future_to_chunk):
                chunk_idx = future_to_chunk[future]
                try:
                    result = future.result()
                    results.append((chunk_idx, result))
                    pbar.update(1)
                except Exception as exc:
                    print(f'Chunk {chunk_idx} ha generato un\'eccezione: {exc}')
                    pbar.update(1)
    
    # Riordina risultati per indice chunk
    results.sort(key=lambda x: x[0])
    all_features = []
    for _, chunk_results in results:
        all_features.extend(chunk_results)
    
    elapsed = time.time() - start_time
    rate = len(series) / elapsed
    print(f"✅ Completato in {elapsed:.1f}s ({rate:,.0f} predizioni/sec)")
    
    return all_features

# Applica l'estrazione delle features con threading
print("🚀 Estraendo features del reasoning con threading (compatibile Jupyter)...")
print(f"📊 Dataset: {len(df_success):,} predizioni con reasoning valido")

start_total = time.time()

# Estrazione con threading
reasoning_features = threaded_feature_extraction(df_success['reason'])
reasoning_df = pd.DataFrame(reasoning_features)

# Combina con i dati originali
print("🔗 Combinando con dati originali...")
df_analysis = pd.concat([df_success.reset_index(drop=True), reasoning_df], axis=1)

total_elapsed = time.time() - start_total
print(f"✅ Feature extraction completata!")
print(f"   • Tempo totale: {total_elapsed:.1f}s")
print(f"   • Rate: {len(df_success)/total_elapsed:,.0f} predizioni/sec")
print(f"   • Reasoning completi: {reasoning_df['reasoning_complete'].sum():,} / {len(reasoning_df):,} ({100*reasoning_df['reasoning_complete'].mean():.1f}%)")

# Memoria utilizzata
memory_usage = df_analysis.memory_usage(deep=True).sum() / 1024**2
print(f"   • Memoria utilizzata: {memory_usage:.1f} MB")

In [ ]:
# Statistiche descrittive del reasoning
reasoning_stats = reasoning_df.describe()
print("Statistiche quantitative del reasoning:")
print(reasoning_stats.round(2))

In [ ]:
# Visualizzazione delle distribuzioni
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Lunghezza in caratteri
axes[0,0].hist(df_analysis['length_chars'], bins=50, alpha=0.7, edgecolor='black')
axes[0,0].set_title('Distribuzione Lunghezza Reasoning (caratteri)')
axes[0,0].set_xlabel('Caratteri')
axes[0,0].set_ylabel('Frequenza')

# Numero di parole
axes[0,1].hist(df_analysis['length_words'], bins=50, alpha=0.7, edgecolor='black')
axes[0,1].set_title('Distribuzione Numero Parole')
axes[0,1].set_xlabel('Parole')
axes[0,1].set_ylabel('Frequenza')

# Numero di frasi
axes[0,2].hist(df_analysis['num_sentences'], bins=20, alpha=0.7, edgecolor='black')
axes[0,2].set_title('Distribuzione Numero Frasi')
axes[0,2].set_xlabel('Frasi')
axes[0,2].set_ylabel('Frequenza')

# Flesch score
axes[1,0].hist(df_analysis[df_analysis['flesch_score'] > 0]['flesch_score'], bins=30, alpha=0.7, edgecolor='black')
axes[1,0].set_title('Distribuzione Flesch Reading Ease')
axes[1,0].set_xlabel('Flesch Score')
axes[1,0].set_ylabel('Frequenza')

# Presenza termini semantici
semantic_terms = ['has_geographic_terms', 'has_temporal_terms', 'has_popularity_terms', 'has_distance_terms']
term_counts = [df_analysis[term].sum() for term in semantic_terms]
term_labels = ['Geographic', 'Temporal', 'Popularity', 'Distance']

axes[1,1].bar(term_labels, term_counts, alpha=0.7)
axes[1,1].set_title('Presenza Termini Semantici')
axes[1,1].set_ylabel('Numero Predizioni')
axes[1,1].tick_params(axis='x', rotation=45)

# Completezza reasoning per strategia
completeness_by_strategy = df_analysis.groupby('strategy')['reasoning_complete'].mean()
axes[1,2].bar(completeness_by_strategy.index, completeness_by_strategy.values, alpha=0.7)
axes[1,2].set_title('Completezza Reasoning per Strategia')
axes[1,2].set_ylabel('Proporzione Reasoning Completi')
axes[1,2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 3. Correlazione Reasoning-Performance

In [ ]:
# Correlazione tra features del reasoning e hit rate
numerical_features = ['length_chars', 'length_words', 'num_sentences', 'avg_word_length', 'flesch_score']
boolean_features = ['has_geographic_terms', 'has_temporal_terms', 'has_popularity_terms', 'has_distance_terms', 'reasoning_complete']

print("Correlazione tra features reasoning e successo predittivo:\n")

# Correlazioni numeriche
for feature in numerical_features:
    correlation = df_analysis[feature].corr(df_analysis['hit'].astype(int))
    print(f"{feature:20s}: {correlation:6.3f}")

print()

# Hit rate per presenza di termini specifici
for feature in boolean_features:
    hit_rate_true = df_analysis[df_analysis[feature] == True]['hit'].mean()
    hit_rate_false = df_analysis[df_analysis[feature] == False]['hit'].mean()
    difference = hit_rate_true - hit_rate_false
    print(f"{feature:20s}: Con={hit_rate_true:.3f} | Senza={hit_rate_false:.3f} | Diff={difference:+.3f}")

In [ ]:
# Boxplot delle performance per presenza di termini chiave
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.ravel()

key_features = ['has_geographic_terms', 'has_temporal_terms', 'has_popularity_terms', 'reasoning_complete']
feature_titles = ['Termini Geografici', 'Termini Temporali', 'Termini Popolarità', 'Reasoning Completo']

for i, (feature, title) in enumerate(zip(key_features, feature_titles)):
    # Crea dati per boxplot
    data_true = df_analysis[df_analysis[feature] == True]['hit'].astype(int)
    data_false = df_analysis[df_analysis[feature] == False]['hit'].astype(int)
    
    # Boxplot
    box_data = [data_false, data_true]
    axes[i].boxplot(box_data, labels=['Assente', 'Presente'])
    axes[i].set_title(f'{title}\n(n_assente={len(data_false)}, n_presente={len(data_true)})')
    axes[i].set_ylabel('Hit (0/1)')
    
    # Aggiungi hit rate medio
    mean_false = data_false.mean()
    mean_true = data_true.mean()
    axes[i].text(1, mean_false + 0.05, f'{mean_false:.3f}', ha='center', fontweight='bold')
    axes[i].text(2, mean_true + 0.05, f'{mean_true:.3f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 4. Analisi Qualitativa dei Patterns

In [ ]:
def extract_reasoning_patterns(reason_text):
    """
    Estrae pattern qualitativi dal reasoning
    """
    if pd.isna(reason_text) or reason_text.strip() == '':
        return []
    
    text = str(reason_text).lower()
    patterns = []
    
    # Pattern di ragionamento geografico
    if any(term in text for term in ['close', 'near', 'distance', 'walking', 'vicinity']):
        patterns.append('proximity_reasoning')
    
    # Pattern di popolarità/attrattività
    if any(term in text for term in ['popular', 'famous', 'well-known', 'major attraction']):
        patterns.append('popularity_reasoning')
    
    # Pattern temporale
    if any(term in text for term in ['time', 'hour', 'morning', 'afternoon', 'evening']):
        patterns.append('temporal_reasoning')
    
    # Pattern di continuità tematica
    if any(term in text for term in ['historical', 'cultural', 'pattern', 'similar', 'typical']):
        patterns.append('thematic_continuity')
    
    # Pattern di esclusione logica
    if any(term in text for term in ['excluding', 'not in', 'avoid', 'already visited']):
        patterns.append('exclusion_logic')
    
    # Pattern di esperienza turistica
    if any(term in text for term in ['tourist', 'visitor', 'experience', 'tour', 'sightseeing']):
        patterns.append('tourist_experience')
    
    return patterns

def process_patterns_chunk_threaded(chunk):
    """
    Processa un chunk di pattern con threading (compatibile Jupyter)
    """
    return [extract_reasoning_patterns(text) for text in chunk]

def threaded_pattern_extraction(series, n_threads=N_CORES, chunk_size=CHUNK_SIZE):
    """
    Estrae pattern con ThreadPoolExecutor (compatibile Jupyter)
    """
    print(f"🔍 Avvio estrazione pattern con threading:")
    print(f"   • Dataset: {len(series):,} reasoning texts")
    print(f"   • Threads: {n_threads}")
    
    start_time = time.time()
    
    # Dividi in chunks
    chunks = [series[i:i+chunk_size] for i in range(0, len(series), chunk_size)]
    print(f"   • Chunks creati: {len(chunks)}")
    
    # Processa con ThreadPoolExecutor
    results = []
    with ThreadPoolExecutor(max_workers=n_threads) as executor:
        future_to_chunk = {executor.submit(process_patterns_chunk_threaded, chunk): i 
                          for i, chunk in enumerate(chunks)}
        
        with tqdm(total=len(chunks), desc="Extracting patterns") as pbar:
            for future in as_completed(future_to_chunk):
                chunk_idx = future_to_chunk[future]
                try:
                    result = future.result()
                    results.append((chunk_idx, result))
                    pbar.update(1)
                except Exception as exc:
                    print(f'Pattern chunk {chunk_idx} errore: {exc}')
                    pbar.update(1)
    
    # Riordina e combina risultati
    results.sort(key=lambda x: x[0])
    all_patterns = []
    for _, chunk_results in results:
        all_patterns.extend(chunk_results)
    
    elapsed = time.time() - start_time
    print(f"✅ Pattern extraction completata in {elapsed:.1f}s")
    
    return pd.Series(all_patterns)

# Applica l'estrazione dei pattern con threading
print("🔍 Estraendo pattern qualitativi con threading (compatibile Jupyter)...")
start_time = time.time()

df_analysis['reasoning_patterns'] = threaded_pattern_extraction(df_analysis['reason'])

# Conta i pattern con monitoring
print("📊 Analizzando distribuzione pattern...")
all_patterns = []
for patterns in tqdm(df_analysis['reasoning_patterns'], desc="Counting patterns"):
    all_patterns.extend(patterns)

pattern_counts = Counter(all_patterns)
elapsed = time.time() - start_time

print(f"✅ Pattern analysis completata in {elapsed:.1f}s")
print(f"📈 Pattern di ragionamento più frequenti:")
for pattern, count in pattern_counts.most_common():
    percentage = 100 * count / len(df_analysis)
    print(f"   {pattern:20s}: {count:8,d} ({percentage:5.1f}%)")

In [ ]:
# Visualizza pattern per strategia
pattern_by_strategy = defaultdict(lambda: defaultdict(int))

for _, row in df_analysis.iterrows():
    strategy = row['strategy']
    for pattern in row['reasoning_patterns']:
        pattern_by_strategy[strategy][pattern] += 1

# Crea DataFrame per visualizzazione
pattern_data = []
for strategy, patterns in pattern_by_strategy.items():
    total_predictions = len(df_analysis[df_analysis['strategy'] == strategy])
    for pattern, count in patterns.items():
        pattern_data.append({
            'strategy': strategy,
            'pattern': pattern,
            'count': count,
            'percentage': 100 * count / total_predictions
        })

pattern_df = pd.DataFrame(pattern_data)

# Heatmap dei pattern per strategia
if not pattern_df.empty:
    pivot_df = pattern_df.pivot(index='pattern', columns='strategy', values='percentage').fillna(0)
    
    plt.figure(figsize=(12, 8))
    sns.heatmap(pivot_df, annot=True, fmt='.1f', cmap='YlOrRd', cbar_kws={'label': 'Percentage'})
    plt.title('Pattern di Ragionamento per Strategia (%)')
    plt.xlabel('Strategia')
    plt.ylabel('Pattern di Ragionamento')
    plt.tight_layout()
    plt.show()

## 5. Confronto tra Modelli e Strategie

In [ ]:
# Analisi per modello
model_reasoning_analysis = df_analysis.groupby('model').agg({
    'hit': 'mean',
    'length_chars': 'mean',
    'length_words': 'mean',
    'num_sentences': 'mean',
    'flesch_score': 'mean',
    'reasoning_complete': 'mean',
    'has_geographic_terms': 'mean',
    'has_temporal_terms': 'mean',
    'has_popularity_terms': 'mean'
}).round(3)

print("Analisi reasoning per modello:")
print(model_reasoning_analysis)

print("\n" + "="*50 + "\n")

# Analisi per strategia
strategy_reasoning_analysis = df_analysis.groupby('strategy').agg({
    'hit': 'mean',
    'length_chars': 'mean',
    'length_words': 'mean',
    'num_sentences': 'mean',
    'flesch_score': 'mean',
    'reasoning_complete': 'mean',
    'has_geographic_terms': 'mean',
    'has_temporal_terms': 'mean',
    'has_popularity_terms': 'mean'
}).round(3)

print("Analisi reasoning per strategia:")
print(strategy_reasoning_analysis)

In [ ]:
# Visualizzazione comparativa
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Hit rate per strategia
strategy_hit_rates = df_analysis.groupby('strategy')['hit'].mean()
axes[0,0].bar(strategy_hit_rates.index, strategy_hit_rates.values, alpha=0.7)
axes[0,0].set_title('Hit Rate per Strategia')
axes[0,0].set_ylabel('Hit Rate')
axes[0,0].tick_params(axis='x', rotation=45)

# Lunghezza reasoning per strategia
df_analysis.boxplot(column='length_chars', by='strategy', ax=axes[0,1])
axes[0,1].set_title('Lunghezza Reasoning per Strategia')
axes[0,1].set_ylabel('Caratteri')
axes[0,1].tick_params(axis='x', rotation=45)

# Completezza reasoning per modello
model_completeness = df_analysis.groupby('model')['reasoning_complete'].mean()
axes[1,0].bar(model_completeness.index, model_completeness.values, alpha=0.7)
axes[1,0].set_title('Completezza Reasoning per Modello')
axes[1,0].set_ylabel('Proporzione Reasoning Completi')
axes[1,0].tick_params(axis='x', rotation=45)

# Presenza termini geografici per strategia
geo_terms_by_strategy = df_analysis.groupby('strategy')['has_geographic_terms'].mean()
axes[1,1].bar(geo_terms_by_strategy.index, geo_terms_by_strategy.values, alpha=0.7)
axes[1,1].set_title('Uso Termini Geografici per Strategia')
axes[1,1].set_ylabel('Proporzione')
axes[1,1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 6. Esempi di Reasoning di Alta/Bassa Qualità

In [ ]:
def display_reasoning_examples(df, title, condition, n_examples=5):
    """
    Mostra esempi di reasoning basati su una condizione
    """
    examples = df[condition].sample(min(n_examples, len(df[condition])))
    
    print(f"\n{'='*20} {title} {'='*20}")
    
    for i, (_, row) in enumerate(examples.iterrows(), 1):
        print(f"\nEsempio {i}:")
        print(f"Modello/Strategia: {row['model']}/{row['strategy']}")
        print(f"Current POI: {row['current_poi']}")
        print(f"Ground Truth: {row['ground_truth']}")
        print(f"Hit: {row['hit']}")
        print(f"Reasoning Length: {row['length_chars']} chars, {row['length_words']} words")
        print(f"Reasoning: {row['reason'][:300]}{'...' if len(row['reason']) > 300 else ''}")
        print("-" * 80)

# Esempi di reasoning di alta qualità (completi + hit)
high_quality = (
    (df_analysis['reasoning_complete'] == True) & 
    (df_analysis['hit'] == True) &
    (df_analysis['length_chars'] > 100)
)

display_reasoning_examples(df_analysis, "REASONING DI ALTA QUALITÀ", high_quality)

# Esempi di reasoning di bassa qualità (incompleti o miss)
low_quality = (
    (df_analysis['reasoning_complete'] == False) | 
    ((df_analysis['hit'] == False) & (df_analysis['length_chars'] < 50))
)

display_reasoning_examples(df_analysis, "REASONING DI BASSA QUALITÀ", low_quality)

# Esempi con termini temporali
temporal_reasoning = df_analysis['has_temporal_terms'] == True

display_reasoning_examples(df_analysis, "REASONING CON ASPETTI TEMPORALI", temporal_reasoning, 3)

## 7. Word Cloud e Analisi Lessicale

In [ ]:
# Word cloud per reasoning dei hit vs miss
if WORDCLOUD_AVAILABLE and NLTK_AVAILABLE:
    hit_reasons = ' '.join(df_analysis[df_analysis['hit'] == True]['reason'].astype(str))
    miss_reasons = ' '.join(df_analysis[df_analysis['hit'] == False]['reason'].astype(str))

    # Stopwords personalizzate
    try:
        custom_stopwords = set(stopwords.words('english') + stopwords.words('italian') + 
                              ['poi', 'pois', 'these', 'the', 'are', 'and', 'or', 'in', 'of', 'to', 'for', 'with', 'by', 'from'])
    except:
        custom_stopwords = set(['poi', 'pois', 'these', 'the', 'are', 'and', 'or', 'in', 'of', 'to', 'for', 'with', 'by', 'from'])

    fig, axes = plt.subplots(1, 2, figsize=(20, 8))

    if hit_reasons.strip():
        wordcloud_hit = WordCloud(width=400, height=300, 
                                  background_color='white', 
                                  stopwords=custom_stopwords,
                                  max_words=50).generate(hit_reasons)
        axes[0].imshow(wordcloud_hit, interpolation='bilinear')
        axes[0].set_title('Word Cloud - Reasoning per HITS')
        axes[0].axis('off')

    if miss_reasons.strip():
        wordcloud_miss = WordCloud(width=400, height=300, 
                                   background_color='white', 
                                   stopwords=custom_stopwords,
                                   max_words=50).generate(miss_reasons)
        axes[1].imshow(wordcloud_miss, interpolation='bilinear')
        axes[1].set_title('Word Cloud - Reasoning per MISSES')
        axes[1].axis('off')

    plt.tight_layout()
    plt.show()
else:
    print("⚠️  WordCloud o NLTK non disponibili - sezione saltata")
    print("Analisi delle parole più frequenti nei HIT:")
    
    # Analisi alternativa senza wordcloud
    hit_texts = df_analysis[df_analysis['hit'] == True]['reason'].astype(str)
    all_hit_words = []
    for text in hit_texts.head(1000):  # Campione per performance
        words = [w.lower() for w in text.split() if len(w) > 3 and w.isalpha()]
        all_hit_words.extend(words)
    
    hit_word_counts = Counter(all_hit_words)
    print("Top 20 parole nei reasoning dei HIT:")
    for word, count in hit_word_counts.most_common(20):
        print(f"{word:15s}: {count:5d}")
    
    print("\nAnalisi delle parole più frequenti nei MISS:")
    miss_texts = df_analysis[df_analysis['hit'] == False]['reason'].astype(str)
    all_miss_words = []
    for text in miss_texts.head(1000):  # Campione per performance
        words = [w.lower() for w in text.split() if len(w) > 3 and w.isalpha()]
        all_miss_words.extend(words)
    
    miss_word_counts = Counter(all_miss_words)
    print("Top 20 parole nei reasoning dei MISS:")
    for word, count in miss_word_counts.most_common(20):
        print(f"{word:15s}: {count:5d}")

## 8. Analisi delle Parole Chiave più Discriminanti

In [ ]:
def tokenize_text_chunk(text_list, chunk_idx=0):
    """
    Tokenizza un chunk di testi in parallelo
    """
    all_words = []
    for text in text_list:
        if NLTK_AVAILABLE:
            try:
                words = [word.lower() for word in word_tokenize(text) if word.isalpha() and len(word) > 2]
            except:
                words = [word.lower() for word in text.split() if word.isalpha() and len(word) > 2]
        else:
            words = [word.lower() for word in text.split() if word.isalpha() and len(word) > 2]
        all_words.extend(words)
    return all_words

def threaded_word_analysis(hit_texts, miss_texts, min_freq=5, sample_size=50000):
    """
    Analizza parole discriminanti con ThreadPoolExecutor (compatibile Jupyter)
    """
    print(f"🔍 Avvio analisi parole discriminanti con threading:")
    print(f"   • HIT texts: {len(hit_texts):,}")
    print(f"   • MISS texts: {len(miss_texts):,}")
    
    # Campionamento intelligente per performance
    if len(hit_texts) > sample_size:
        hit_sample = hit_texts.sample(sample_size, random_state=42)
        print(f"   • HIT campionati: {len(hit_sample):,}")
    else:
        hit_sample = hit_texts
        
    if len(miss_texts) > sample_size:
        miss_sample = miss_texts.sample(sample_size, random_state=42)
        print(f"   • MISS campionati: {len(miss_sample):,}")
    else:
        miss_sample = miss_texts
    
    start_time = time.time()
    
    # Processa HIT texts con ThreadPoolExecutor
    print("📝 Tokenizzando HIT texts con threading...")
    hit_chunks = [hit_sample[i:i+CHUNK_SIZE].tolist() 
                  for i in range(0, len(hit_sample), CHUNK_SIZE)]
    
    hit_words = []
    with ThreadPoolExecutor(max_workers=N_CORES) as executor:
        hit_futures = [executor.submit(tokenize_text_chunk, chunk, i) 
                      for i, chunk in enumerate(hit_chunks)]
        
        for future in tqdm(as_completed(hit_futures), total=len(hit_futures), desc="HIT tokenization"):
            try:
                result = future.result()
                hit_words.extend(result)
            except Exception as e:
                print(f"Errore processing HIT chunk: {e}")
    
    # Processa MISS texts con ThreadPoolExecutor
    print("📝 Tokenizzando MISS texts con threading...")
    miss_chunks = [miss_sample[i:i+CHUNK_SIZE].tolist() 
                   for i in range(0, len(miss_sample), CHUNK_SIZE)]
    
    miss_words = []
    with ThreadPoolExecutor(max_workers=N_CORES) as executor:
        miss_futures = [executor.submit(tokenize_text_chunk, chunk, i) 
                       for i, chunk in enumerate(miss_chunks)]
        
        for future in tqdm(as_completed(miss_futures), total=len(miss_futures), desc="MISS tokenization"):
            try:
                result = future.result()
                miss_words.extend(result)
            except Exception as e:
                print(f"Errore processing MISS chunk: {e}")
    
    # Analisi discriminante
    print("📊 Calcolando scores discriminanti...")
    hit_counter = Counter(hit_words)
    miss_counter = Counter(miss_words)
    
    common_words = set(hit_counter.keys()) & set(miss_counter.keys())
    print(f"   • Parole comuni: {len(common_words):,}")
    
    discriminant_scores = []
    for word in tqdm(common_words, desc="Computing discriminant scores"):
        hit_freq = hit_counter[word]
        miss_freq = miss_counter[word]
        total_freq = hit_freq + miss_freq
        
        if total_freq >= min_freq:
            hit_rate = hit_freq / len(hit_sample)
            miss_rate = miss_freq / len(miss_sample)
            
            if hit_rate + miss_rate > 0:
                score = (hit_rate - miss_rate) / (hit_rate + miss_rate)
                discriminant_scores.append({
                    'word': word,
                    'hit_freq': hit_freq,
                    'miss_freq': miss_freq,
                    'total_freq': total_freq,
                    'discriminant_score': score
                })
    
    elapsed = time.time() - start_time
    print(f"✅ Analisi completata in {elapsed:.1f}s")
    print(f"   • Parole discriminanti trovate: {len(discriminant_scores):,}")
    
    discriminant_df = pd.DataFrame(discriminant_scores)
    return discriminant_df.sort_values('discriminant_score', key=abs, ascending=False)

def analyze_discriminant_words(df, min_freq=5):
    """
    Trova le parole più discriminanti tra hit e miss con ottimizzazioni
    """
    hit_texts = df[df['hit'] == True]['reason'].astype(str)
    miss_texts = df[df['hit'] == False]['reason'].astype(str)
    
    return threaded_word_analysis(hit_texts, miss_texts, min_freq)

# Analizza parole discriminanti con threading ottimizzato
print("🔍 Analizzando parole discriminanti con threading (compatibile Jupyter)...")
discriminant_words = analyze_discriminant_words(df_analysis)

if len(discriminant_words) > 0:
    print("\n📈 Top 20 parole più discriminanti (positive = più presenti nei HIT):")
    print(discriminant_words.head(20).round(3))

    # Visualizzazione ottimizzata
    top_words = discriminant_words.head(15)
    
    if len(top_words) > 0:
        plt.figure(figsize=(12, 8))
        colors = ['green' if score > 0 else 'red' for score in top_words['discriminant_score']]
        plt.barh(top_words['word'], top_words['discriminant_score'], color=colors, alpha=0.7)
        plt.xlabel('Discriminant Score (Positivo = più nei HIT, Negativo = più nei MISS)')
        plt.title('Parole più Discriminanti nei Reasoning')
        plt.axvline(x=0, color='black', linestyle='-', alpha=0.3)
        plt.tight_layout()
        plt.show()
    else:
        print("⚠️  Nessuna parola discriminante visualizzabile")
else:
    print("⚠️  Nessuna parola discriminante trovata")

## 9. Summary e Insights

In [ ]:
def generate_insights_summary(df):
    """
    Genera un riassunto degli insights principali
    """
    insights = []
    
    # Performance generale
    overall_hit_rate = df['hit'].mean()
    insights.append(f"Hit rate complessivo: {overall_hit_rate:.3f} ({100*overall_hit_rate:.1f}%)")
    
    # Correlazione lunghezza-performance
    length_corr = df['length_chars'].corr(df['hit'].astype(int))
    insights.append(f"Correlazione lunghezza-performance: {length_corr:.3f}")
    
    # Migliore strategia
    best_strategy = df.groupby('strategy')['hit'].mean().idxmax()
    best_hit_rate = df.groupby('strategy')['hit'].mean().max()
    insights.append(f"Migliore strategia: {best_strategy} ({100*best_hit_rate:.1f}% hit rate)")
    
    # Termini più utili
    geo_improvement = (df[df['has_geographic_terms']]['hit'].mean() - 
                      df[~df['has_geographic_terms']]['hit'].mean())
    temp_improvement = (df[df['has_temporal_terms']]['hit'].mean() - 
                       df[~df['has_temporal_terms']]['hit'].mean())
    pop_improvement = (df[df['has_popularity_terms']]['hit'].mean() - 
                      df[~df['has_popularity_terms']]['hit'].mean())
    
    insights.append(f"Miglioramento con termini geografici: {100*geo_improvement:+.1f}%")
    insights.append(f"Miglioramento con termini temporali: {100*temp_improvement:+.1f}%")
    insights.append(f"Miglioramento con termini di popolarità: {100*pop_improvement:+.1f}%")
    
    # Completezza reasoning
    complete_hit_rate = df[df['reasoning_complete']]['hit'].mean()
    incomplete_hit_rate = df[~df['reasoning_complete']]['hit'].mean()
    completeness_improvement = complete_hit_rate - incomplete_hit_rate
    insights.append(f"Miglioramento con reasoning completo: {100*completeness_improvement:+.1f}%")
    
    return insights

# Genera insights
insights = generate_insights_summary(df_analysis)

print("\n" + "="*60)
print("                    INSIGHTS PRINCIPALI")
print("="*60)

for i, insight in enumerate(insights, 1):
    print(f"{i:2d}. {insight}")

print("\n" + "="*60)
print("                       RACCOMANDAZIONI")
print("="*60)

recommendations = [
    "Favorire reasoning più lunghi e dettagliati",
    "Includere sempre termini geografici nel reasoning",
    "Utilizzare informazioni temporali quando disponibili",
    "Assicurarsi che il reasoning sia completo e non troncato",
    "La strategia 'with_geom_time' sembra più promettente",
    "Evitare reasoning troppo generici o ripetitivi"
]

for i, rec in enumerate(recommendations, 1):
    print(f"{i:2d}. {rec}")

print("\n" + "="*60)

In [ ]:
# Salva risultati per analisi future con ottimizzazioni memoria
print("💾 Salvando risultati con ottimizzazioni memoria...")
start_time = time.time()

# Calcola statistics in chunks per ridurre memoria
def calculate_summary_stats_chunked(df, chunk_size=50000):
    """
    Calcola statistiche riepilogative in chunks per ottimizzare memoria
    """
    print(f"📊 Calcolando statistiche riepilogative in chunks di {chunk_size:,}...")
    
    # Raggruppa per modello/strategia
    groups = df.groupby(['model', 'strategy'])
    summary_data = []
    
    for (model, strategy), group in tqdm(groups, desc="Processing model/strategy groups"):
        group_stats = {
            'model': model,
            'strategy': strategy,
            'total_predictions': len(group),
            'hit_rate': group['hit'].mean(),
            'avg_length_chars': group['length_chars'].mean(),
            'avg_length_words': group['length_words'].mean(),
            'complete_reasoning_rate': group['reasoning_complete'].mean(),
            'geo_terms_rate': group['has_geographic_terms'].mean(),
            'temporal_terms_rate': group['has_temporal_terms'].mean(),
            'popularity_terms_rate': group['has_popularity_terms'].mean()
        }
        summary_data.append(group_stats)
    
    return pd.DataFrame(summary_data).round(3)

# Calcola statistiche con chunk processing
summary_stats = calculate_summary_stats_chunked(df_analysis)

# Salva con compressione
output_file = '../results/reasoning_analysis_summary.csv'
summary_stats.to_csv(output_file, index=False, compression='gzip')

# Statistiche memoria e performance
elapsed = time.time() - start_time
final_memory = df_analysis.memory_usage(deep=True).sum() / 1024**2
final_rows = len(df_analysis)

print(f"✅ Salvataggio completato in {elapsed:.1f}s")
print(f"   • File salvato: {output_file}")
print(f"   • Dimensioni finali: {final_rows:,} righe")
print(f"   • Memoria utilizzata: {final_memory:.1f} MB")
print(f"   • Memoria per riga: {final_memory/final_rows*1024:.1f} KB/riga")

# Cleanup opzionale per liberare memoria
print("\n🧹 Cleanup memoria opzionale:")
print("   Per liberare memoria, esegui: del df_success, reasoning_df")

# Mostra utilizzo finale con fallback
if PSUTIL_AVAILABLE:
    try:
        process = psutil.Process()
        memory_info = process.memory_info()
        print(f"   • RAM processo: {memory_info.rss / 1024**2:.1f} MB")
        print(f"   • RAM virtuale: {memory_info.vms / 1024**2:.1f} MB")
    except:
        print("   • Monitoraggio RAM non disponibile")
else:
    print("   • PSUtil non disponibile - monitoraggio RAM saltato")

print(f"\n✅ Analisi reasoning completata su {final_rows:,} predizioni!")

In [ ]:
def export_for_canva():
    """
    Esporta tutti i dati in formato CSV ottimizzato per Canva
    """
    print("📊 Esportando dati per Canva...")
    export_dir = "reason"
    
    # Crea timestamp per versionamento
    from datetime import datetime
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    exported_files = []
    
    # 1. PERFORMANCE PER MODELLO - Bar Chart
    print("   📈 1. Performance per modello...")
    model_performance = df_analysis.groupby('model').agg({
        'hit': 'mean',
        'length_chars': 'mean',
        'reasoning_complete': 'mean'
    }).round(3)
    model_performance.columns = ['Hit_Rate', 'Avg_Length_Chars', 'Complete_Rate']
    model_performance['Model'] = model_performance.index
    model_performance = model_performance[['Model', 'Hit_Rate', 'Avg_Length_Chars', 'Complete_Rate']]
    
    file_path = f"{export_dir}/01_model_performance_{timestamp}.csv"
    model_performance.to_csv(file_path, index=False)
    exported_files.append(file_path)
    
    # 2. PERFORMANCE PER STRATEGIA - Bar Chart  
    print("   📈 2. Performance per strategia...")
    strategy_performance = df_analysis.groupby('strategy').agg({
        'hit': 'mean',
        'length_chars': 'mean', 
        'reasoning_complete': 'mean'
    }).round(3)
    strategy_performance.columns = ['Hit_Rate', 'Avg_Length_Chars', 'Complete_Rate']
    strategy_performance['Strategy'] = strategy_performance.index
    strategy_performance = strategy_performance[['Strategy', 'Hit_Rate', 'Avg_Length_Chars', 'Complete_Rate']]
    
    file_path = f"{export_dir}/02_strategy_performance_{timestamp}.csv"
    strategy_performance.to_csv(file_path, index=False)
    exported_files.append(file_path)
    
    # 3. IMPATTO TERMINI SEMANTICI - Horizontal Bar Chart
    print("   📊 3. Impatto termini semantici...")
    semantic_impact = []
    features = ['has_geographic_terms', 'has_temporal_terms', 'has_popularity_terms', 'has_distance_terms']
    labels = ['Geographic Terms', 'Temporal Terms', 'Popularity Terms', 'Distance Terms']
    
    for feature, label in zip(features, labels):
        with_term = df_analysis[df_analysis[feature] == True]['hit'].mean()
        without_term = df_analysis[df_analysis[feature] == False]['hit'].mean()
        improvement = with_term - without_term
        
        semantic_impact.append({
            'Term_Type': label,
            'Hit_Rate_With': round(with_term, 3),
            'Hit_Rate_Without': round(without_term, 3),
            'Improvement': round(improvement, 3),
            'Improvement_Percent': round(improvement * 100, 1)
        })
    
    semantic_df = pd.DataFrame(semantic_impact)
    file_path = f"{export_dir}/03_semantic_terms_impact_{timestamp}.csv"
    semantic_df.to_csv(file_path, index=False)
    exported_files.append(file_path)
    
    # 4. DISTRIBUZIONE LUNGHEZZA REASONING - Histogram Data
    print("   📊 4. Distribuzione lunghezza reasoning...")
    # Crea bins per istogramma
    bins = [0, 50, 100, 200, 300, 500, 1000, 2000, float('inf')]
    labels = ['0-50', '51-100', '101-200', '201-300', '301-500', '501-1000', '1001-2000', '2000+']
    
    df_analysis['length_bin'] = pd.cut(df_analysis['length_chars'], bins=bins, labels=labels, right=False)
    length_dist = df_analysis.groupby('length_bin').agg({
        'hit': ['count', 'mean']
    }).round(3)
    length_dist.columns = ['Count', 'Hit_Rate']
    length_dist['Length_Range'] = length_dist.index
    length_dist = length_dist[['Length_Range', 'Count', 'Hit_Rate']].reset_index(drop=True)
    
    file_path = f"{export_dir}/04_length_distribution_{timestamp}.csv"
    length_dist.to_csv(file_path, index=False)
    exported_files.append(file_path)
    
    # 5. TOP PATTERN DI REASONING - Pie Chart
    print("   📊 5. Pattern di reasoning...")
    if len(pattern_counts) > 0:
        pattern_data = []
        total_patterns = sum(pattern_counts.values())
        
        for pattern, count in pattern_counts.most_common(6):  # Top 6 per leggibilità
            percentage = (count / total_patterns) * 100
            pattern_data.append({
                'Pattern': pattern.replace('_', ' ').title(),
                'Count': count,
                'Percentage': round(percentage, 1)
            })
        
        pattern_df = pd.DataFrame(pattern_data)
        file_path = f"{export_dir}/05_reasoning_patterns_{timestamp}.csv"
        pattern_df.to_csv(file_path, index=False)
        exported_files.append(file_path)
    
    # 6. MATRICE MODELLO-STRATEGIA - Heatmap
    print("   📊 6. Matrice modello-strategia...")
    model_strategy_matrix = df_analysis.groupby(['model', 'strategy'])['hit'].mean().round(3).reset_index()
    model_strategy_matrix.columns = ['Model', 'Strategy', 'Hit_Rate']
    
    file_path = f"{export_dir}/06_model_strategy_matrix_{timestamp}.csv"
    model_strategy_matrix.to_csv(file_path, index=False)
    exported_files.append(file_path)
    
    # 7. TOP PAROLE DISCRIMINANTI - Word Cloud Data
    print("   📊 7. Parole discriminanti...")
    if len(discriminant_words) > 0:
        top_discriminant = discriminant_words.head(20).copy()
        top_discriminant['Word'] = top_discriminant['word']
        top_discriminant['Score'] = top_discriminant['discriminant_score'].round(3)
        top_discriminant['Type'] = top_discriminant['Score'].apply(lambda x: 'Positive (HIT)' if x > 0 else 'Negative (MISS)')
        top_discriminant['Abs_Score'] = abs(top_discriminant['Score'])
        
        discriminant_export = top_discriminant[['Word', 'Score', 'Type', 'Abs_Score', 'hit_freq', 'miss_freq']]
        file_path = f"{export_dir}/07_discriminant_words_{timestamp}.csv"
        discriminant_export.to_csv(file_path, index=False)
        exported_files.append(file_path)
    
    # 8. SUMMARY INSIGHTS - Text Data per Infografica
    print("   📊 8. Summary insights...")
    insights_data = {
        'Metric': [
            'Total Predictions Analyzed',
            'Overall Hit Rate',
            'Best Strategy',
            'Best Model',
            'Geographic Terms Improvement',
            'Temporal Terms Improvement',
            'Complete Reasoning Improvement',
            'Average Reasoning Length',
            'Most Common Pattern'
        ],
        'Value': [
            f"{len(df_analysis):,}",
            f"{df_analysis['hit'].mean():.1%}",
            f"{df_analysis.groupby('strategy')['hit'].mean().idxmax()}",
            f"{df_analysis.groupby('model')['hit'].mean().idxmax()}",
            f"+{(df_analysis[df_analysis['has_geographic_terms']]['hit'].mean() - df_analysis[~df_analysis['has_geographic_terms']]['hit'].mean())*100:.1f}%",
            f"+{(df_analysis[df_analysis['has_temporal_terms']]['hit'].mean() - df_analysis[~df_analysis['has_temporal_terms']]['hit'].mean())*100:.1f}%",
            f"+{(df_analysis[df_analysis['reasoning_complete']]['hit'].mean() - df_analysis[~df_analysis['reasoning_complete']]['hit'].mean())*100:.1f}%",
            f"{df_analysis['length_chars'].mean():.0f} chars",
            f"{pattern_counts.most_common(1)[0][0].replace('_', ' ').title()}" if pattern_counts else "N/A"
        ],
        'Category': [
            'Volume', 'Performance', 'Strategy', 'Model', 'Features', 'Features', 'Quality', 'Quality', 'Patterns'
        ]
    }
    
    insights_df = pd.DataFrame(insights_data)
    file_path = f"{export_dir}/08_summary_insights_{timestamp}.csv"
    insights_df.to_csv(file_path, index=False)
    exported_files.append(file_path)
    
    # 9. CORRELAZIONI - Scatter Plot Data
    print("   📊 9. Correlazioni...")
    correlation_data = []
    numerical_features = ['length_chars', 'length_words', 'num_sentences', 'avg_word_length']
    
    for feature in numerical_features:
        correlation = df_analysis[feature].corr(df_analysis['hit'].astype(int))
        correlation_data.append({
            'Feature': feature.replace('_', ' ').title(),
            'Correlation_with_Hit_Rate': round(correlation, 3),
            'Correlation_Strength': 'Strong' if abs(correlation) > 0.3 else 'Moderate' if abs(correlation) > 0.1 else 'Weak'
        })
    
    correlation_df = pd.DataFrame(correlation_data)
    file_path = f"{export_dir}/09_correlations_{timestamp}.csv"
    correlation_df.to_csv(file_path, index=False)
    exported_files.append(file_path)
    
    return exported_files

# Esegui export per Canva
print("🎨 EXPORT PER CANVA - REASONING ANALYSIS")
print("=" * 50)

exported_files = export_for_canva()

print(f"\n✅ Export completato! File creati:")
for i, file_path in enumerate(exported_files, 1):
    file_size = os.path.getsize(file_path) / 1024  # KB
    print(f"   {i:2d}. {file_path} ({file_size:.1f} KB)")

print(f"\n📁 Tutti i file salvati in: notebook/reason/")
print(f"📊 {len(exported_files)} CSV pronti per import in Canva")

# Genera README per Canva
readme_content = f"""# Reasoning Analysis - CSV Export per Canva

## File Esportati ({len(exported_files)} files):

### 📊 GRAFICI RACCOMANDATI PER PRESENTAZIONE:

1. **01_model_performance_*.csv** 
   - GRAFICO: Bar Chart (Verticale)
   - X-axis: Model, Y-axis: Hit_Rate
   - Colori: Gradiente blu/verde per performance

2. **02_strategy_performance_*.csv**
   - GRAFICO: Bar Chart (Orizzontale) 
   - X-axis: Hit_Rate, Y-axis: Strategy
   - Evidenzia 'with_geom_time' come migliore

3. **03_semantic_terms_impact_*.csv**
   - GRAFICO: Horizontal Bar Chart (Before/After)
   - Mostra Hit_Rate_Without vs Hit_Rate_With
   - Colori: Rosso (senza) vs Verde (con)

4. **04_length_distribution_*.csv**
   - GRAFICO: Histogram + Line Chart (Hit Rate)
   - Dual axis: Count (bars) + Hit_Rate (line)

5. **05_reasoning_patterns_*.csv**
   - GRAFICO: Pie Chart o Donut Chart
   - Percentages con labels pattern

6. **06_model_strategy_matrix_*.csv**
   - GRAFICO: Heatmap Matrix
   - Pivot: Model (rows) x Strategy (columns)
   - Color scale per Hit_Rate

7. **07_discriminant_words_*.csv**
   - GRAFICO: Word Cloud o Horizontal Bar
   - Size/length basato su Abs_Score
   - Colori: Verde (Positive) vs Rosso (Negative)

8. **08_summary_insights_*.csv**
   - GRAFICO: Infographic Cards
   - Key metrics in evidenza
   - Categorize per colore

9. **09_correlations_*.csv**
   - GRAFICO: Scatter Plot o Bar Chart
   - Correlation_with_Hit_Rate (Y-axis)
   - Feature (X-axis)

## 🎨 CONSIGLI DESIGN CANVA:
- Usa palette colori coerente (blu/verde per positivo, rosso per negativo)
- Font sans-serif per leggibilità
- Titoli accattivanti per ogni grafico
- Include sempre source: "LLM-Tourism-Mobility Analysis"
"""

readme_path = f"reason/README_canva_export.md"
with open(readme_path, 'w', encoding='utf-8') as f:
    f.write(readme_content)

print(f"📋 README creato: {readme_path}")
print("\n🎨 Pronto per Canva! Import i CSV e segui le raccomandazioni grafiche nel README.")

## 10. Export CSV per Canva - Visualizzazioni Presentazione

Questa sezione esporta tutti i dati di analisi in formato CSV ottimizzato per importazione diretta in Canva per la creazione di visualizzazioni professionali per presentazioni.